In [1]:
import joblib
features = ["temperature_2m_mean",
    "temperature_2m_min",
    "temperature_2m_max",
    "wet_bulb_temperature_2m_mean",
    "dew_point_2m_mean",
    "relative_humidity_2m_mean",
    "cloud_cover_mean",
    "wind_speed_10m_mean",
    "pressure_msl_mean",
    "precipitation_sum",
    "snowfall_sum"]
joblib.dump(features, "./features.pkl")

['./features.pkl']

In [2]:
import requests
import pandas as pd
import joblib

url = "https://archive-api.open-meteo.com/v1/archive"

from datetime import datetime
from zoneinfo import ZoneInfo

BERLIN_TZ = ZoneInfo("Europe/Berlin")

today = datetime.now(BERLIN_TZ).date()

daily_vars = joblib.load("./features.pkl")

params = {
    "latitude": 51.5,
    "longitude": 10.5,
    "start_date": "2021-01-01",
    "end_date": today,
    "daily": ",".join(daily_vars),
    "timezone": "auto",
}

r = requests.get(url, params=params, timeout=30)
print(r.status_code, r.text[:300])  # helpful if it fails again
r.raise_for_status()

data = r.json()
df = pd.DataFrame(data["daily"])
#print(df.head())

#df = df.dropna(subset=features)

#df.head()

df["time"] = pd.to_datetime(df["time"])

df["rain_binary"] = (df["precipitation_sum"] > 0).astype(int)

df["snow_binary"] = (df["snowfall_sum"] > 0).astype(int)

needed_cols = ["time"] + features + ["rain_binary", "snow_binary"]
needed_cols = list(dict.fromkeys(needed_cols))  # unique while preserving order

df = df[needed_cols].dropna().sort_values("time").reset_index(drop=True)

df["time"] = df["time"].dt.date

df.head()


200 {"latitude":51.493847,"longitude":10.434783,"generationtime_ms":222.34642505645752,"utc_offset_seconds":3600,"timezone":"Europe/Berlin","timezone_abbreviation":"GMT+1","elevation":309.0,"daily_units":{"time":"iso8601","temperature_2m_mean":"°C","temperature_2m_min":"°C","temperature_2m_max":"°C","we


,time,temperature_2m_mean,temperature_2m_min,temperature_2m_max,wet_bulb_temperature_2m_mean,dew_point_2m_mean,relative_humidity_2m_mean,cloud_cover_mean,wind_speed_10m_mean,pressure_msl_mean,precipitation_sum,snowfall_sum,rain_binary,snow_binary
0,2021-01-01,0.6,-1.3,1.8,-0.0,-0.3,94,84,5.4,1008.2,1.7,0.63,1,1
1,2021-01-02,-0.5,-3.2,2.7,-1.6,-2.2,88,39,5.0,1014.3,0.0,0.00,0,0
2,2021-01-03,0.2,-0.5,1.0,-1.0,-1.9,86,99,11.5,1014.1,2.9,2.03,1,1
3,2021-01-04,-0.3,-0.8,0.4,-1.4,-2.1,88,99,8.6,1016.5,1.6,1.05,1,1
4,2021-01-05,0.2,-0.6,1.7,-0.8,-1.3,90,99,6.5,1016.5,2.0,1.40,1,1


In [3]:
import requests
import pandas as pd

url = "https://archive-api.open-meteo.com/v1/archive"

from datetime import datetime
from zoneinfo import ZoneInfo

BERLIN_TZ = ZoneInfo("Europe/Berlin")

today = datetime.now(BERLIN_TZ).date()



daily_vars = [
    "temperature_2m_mean",
    "temperature_2m_min",
    "temperature_2m_max",
    "wet_bulb_temperature_2m_mean",
    "dew_point_2m_mean",
    "relative_humidity_2m_mean",
    "cloud_cover_mean",
    "wind_speed_10m_mean",
    "pressure_msl_mean",
    "visibility_mean",
    "updraft_max",
    "precipitation_sum",
    "snowfall_sum"   # <-- keep this as your TARGET later
]

params = {
    "latitude": 51.5,
    "longitude": 10.5,
    "start_date": "2021-01-01",
    "end_date": today,
    "daily": ",".join(daily_vars),
    "timezone": "auto",
}

r = requests.get(url, params=params, timeout=30)
print(r.status_code, r.text[:300])  # helpful if it fails again
r.raise_for_status()

data = r.json()
df = pd.DataFrame(data["daily"])
print(df.head())
print(data.get("daily_units"))
print(df.shape)


200 {"latitude":51.493847,"longitude":10.434783,"generationtime_ms":12.180566787719727,"utc_offset_seconds":3600,"timezone":"Europe/Berlin","timezone_abbreviation":"GMT+1","elevation":309.0,"daily_units":{"time":"iso8601","temperature_2m_mean":"°C","temperature_2m_min":"°C","temperature_2m_max":"°C","we
         time  temperature_2m_mean  temperature_2m_min  temperature_2m_max  \
0  2021-01-01                  0.6                -1.3                 1.8   
1  2021-01-02                 -0.5                -3.2                 2.7   
2  2021-01-03                  0.2                -0.5                 1.0   
3  2021-01-04                 -0.3                -0.8                 0.4   
4  2021-01-05                  0.2                -0.6                 1.7   

   wet_bulb_temperature_2m_mean  dew_point_2m_mean  relative_humidity_2m_mean  \
0                          -0.0               -0.3                         94   
1                          -1.6               -2.2              

In [4]:
import numpy as np
import pandas as pd

# Convert "None" -> NaN (works for object columns)
df = df.replace({None: np.nan})

# Convert columns (except time) to numeric
for c in df.columns:
    if c != "time":
        df[c] = pd.to_numeric(df[c], errors="coerce")


/var/folders/vf/zrflmsnd3ns54k88g4q54x2r0000gn/T/ipykernel_1392/1191195721.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({None: np.nan})


In [5]:
df = df.drop(columns=["visibility_mean", "updraft_max"], errors="ignore")
features = [
    "temperature_2m_mean",
    "temperature_2m_min",
    "temperature_2m_max",
    "wet_bulb_temperature_2m_mean",
    "dew_point_2m_mean",
    "relative_humidity_2m_mean",
    "cloud_cover_mean",
    "wind_speed_10m_mean",
    "pressure_msl_mean",
    "precipitation_sum"
]

target = "snowfall_sum"

df2 = df.dropna(subset=features + [target])

print("Rows after cleaning:", len(df2))
print(df2.head())


Rows after cleaning: 1862
         time  temperature_2m_mean  temperature_2m_min  temperature_2m_max  \
0  2021-01-01                  0.6                -1.3                 1.8   
1  2021-01-02                 -0.5                -3.2                 2.7   
2  2021-01-03                  0.2                -0.5                 1.0   
3  2021-01-04                 -0.3                -0.8                 0.4   
4  2021-01-05                  0.2                -0.6                 1.7   

   wet_bulb_temperature_2m_mean  dew_point_2m_mean  relative_humidity_2m_mean  \
0                          -0.0               -0.3                         94   
1                          -1.6               -2.2                         88   
2                          -1.0               -1.9                         86   
3                          -1.4               -2.1                         88   
4                          -0.8               -1.3                         90   

   cloud_cover_mea

In [6]:
import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# ----------------------------
# 0) Ensure df2 is ready
# df2 must contain: time, precipitation_sum, snowfall_sum and all feature cols
# ----------------------------
df2 = df2.copy()
df2["time"] = pd.to_datetime(df2["time"])

# ----------------------------
# 1) Feature lists (Rain + Snow)
# ----------------------------
rain_features = [
    "temperature_2m_mean",
    "temperature_2m_min",
    "temperature_2m_max",
    "wet_bulb_temperature_2m_mean",
    "dew_point_2m_mean",
    "relative_humidity_2m_mean",
    "cloud_cover_mean",
    "wind_speed_10m_mean",
    "pressure_msl_mean"
]

# If snow uses the same features, keep it.
# If you want different features for snow, replace here.
snow_features = rain_features.copy()

# ----------------------------
# 2) Create binary targets
# ----------------------------
df2["rain_binary"] = (df2["precipitation_sum"] > 0).astype(int)
df2["snow_binary"] = (df2["snowfall_sum"] > 0).astype(int)

# ----------------------------
# 3) Drop rows with missing values in required columns
# ----------------------------
needed_cols = ["time"] + rain_features + snow_features + ["rain_binary", "snow_binary"]
needed_cols = list(dict.fromkeys(needed_cols))  # unique while preserving order
df2_clean = df2[needed_cols].dropna().sort_values("time").reset_index(drop=True)

print("Rows after cleaning:", len(df2_clean))
print("Rain positives:", df2_clean["rain_binary"].sum(), "/", len(df2_clean))
print("Snow positives:", df2_clean["snow_binary"].sum(), "/", len(df2_clean))

# ----------------------------
# 4) Time-based split (80/20)
# ----------------------------
split_idx = int(len(df2_clean) * 0.8)
train_df = df2_clean.iloc[:split_idx]
test_df  = df2_clean.iloc[split_idx:]

# ----------------------------
# 5) Train RAIN model (Logistic Regression pipeline)
# ----------------------------
X_train_rain = train_df[rain_features]
y_train_rain = train_df["rain_binary"]

X_test_rain = test_df[rain_features]
y_test_rain = test_df["rain_binary"]

rain_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        class_weight="balanced"  # good for imbalance; remove if you don't want
    ))
])

rain_pipe.fit(X_train_rain, y_train_rain)

rain_prob = rain_pipe.predict_proba(X_test_rain)[:, 1]
rain_pred = (rain_prob > 0.5).astype(int)

print("\n====== RAIN MODEL ======")
print(classification_report(y_test_rain, rain_pred))
print("ROC-AUC:", roc_auc_score(y_test_rain, rain_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test_rain, rain_pred))

# ----------------------------
# 6) Train SNOW model (your exact pipeline)
# ----------------------------
X_train_snow = train_df[snow_features]
y_train_snow = train_df["snow_binary"]

X_test_snow = test_df[snow_features]
y_test_snow = test_df["snow_binary"]

snow_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=5000,
        solver="lbfgs"
    ))
])

snow_pipe.fit(X_train_snow, y_train_snow)

snow_prob = snow_pipe.predict_proba(X_test_snow)[:, 1]
snow_pred = (snow_prob > 0.5).astype(int)

print("\n====== SNOW MODEL ======")
print(classification_report(y_test_snow, snow_pred))
print("ROC-AUC:", roc_auc_score(y_test_snow, snow_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test_snow, snow_pred))

# ----------------------------
# 7) Save models + data for Streamlit
# ----------------------------
joblib.dump(rain_pipe, "./rain_pipe.pkl")
joblib.dump(snow_pipe, "./snow_pipe.pkl")

# Save df2 for date-based lookup in Streamlit
df2_clean_out = df2_clean.copy()
df2_clean_out["time"] = df2_clean_out["time"].dt.date  # simpler for Streamlit date compare
df2_clean_out.to_csv("df2.csv", index=False)

# Save feature lists (optional but very useful)
joblib.dump(rain_features, "./rain_features.pkl")
joblib.dump(snow_features, "./snow_features.pkl")

print("\n✅ Saved:")
print(" - rain_pipe.pkl")
print(" - snow_pipe.pkl")
print(" - df2.csv")
print(" - rain_features.pkl")
print(" - snow_features.pkl")

Rows after cleaning: 1862
Rain positives: 1188 / 1862
Snow positives: 234 / 1862

====== RAIN MODEL ======
              precision    recall  f1-score   support

           0       0.75      0.84      0.80       161
           1       0.87      0.79      0.83       212

    accuracy                           0.81       373
   macro avg       0.81      0.82      0.81       373
weighted avg       0.82      0.81      0.81       373

ROC-AUC: 0.8967830774639635
Confusion Matrix:
 [[136  25]
 [ 45 167]]

====== SNOW MODEL ======
              precision    recall  f1-score   support

           0       0.97      0.96      0.96       335
           1       0.68      0.71      0.69        38

    accuracy                           0.94       373
   macro avg       0.82      0.84      0.83       373
weighted avg       0.94      0.94      0.94       373

ROC-AUC: 0.9671641791044776
Confusion Matrix:
 [[322  13]
 [ 11  27]]

✅ Saved:
 - rain_pipe.pkl
 - snow_pipe.pkl
 - df2.csv
 - rain_features.p

In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, roc_curve

prec_rain = precision_score(y_test_rain, rain_pred)
rec_rain  = recall_score(y_test_rain, rain_pred)
f1_rain   = f1_score(y_test_rain, rain_pred)
acc_rain  = accuracy_score(y_test_rain, rain_pred)
p_rain = rain_pipe.predict_proba(X_test_rain)[:, 1]
fpr_rain, tpr_rain, _ = roc_curve(y_test_rain, p_rain)
auc_rain = roc_auc_score(y_test_rain, p_rain)


prec_snow = precision_score(y_test_snow, snow_pred)
rec_snow  = recall_score(y_test_snow, snow_pred)
f1_snow   = f1_score(y_test_snow, snow_pred)
acc_snow  = accuracy_score(y_test_snow, snow_pred)
p_snow = snow_pipe.predict_proba(X_test_snow)[:, 1]
fpr_snow, tpr_snow, _ = roc_curve(y_test_snow, p_snow)
auc_snow = roc_auc_score(y_test_snow, p_snow)

In [8]:
import joblib

joblib.dump(
    {
        "prec_rain": prec_rain, "rec_rain": rec_rain, "f1_rain": f1_rain, "acc_rain": acc_rain,
        "prec_snow": prec_snow, "rec_snow": rec_snow, "f1_snow": f1_snow, "acc_snow": acc_snow,
        "p_rain": p_rain, "fpr_rain": fpr_rain, "auc_rain": auc_rain, "tpr_rain": tpr_rain,
        "p_snow": p_snow, "fpr_snow": fpr_snow, "auc_snow": auc_snow, "tpr_snow": tpr_snow
    },
    "./metrics.pkl"
)


['./metrics.pkl']